# Week 1 — Day 1: LLM API Fundamentals

## Goals

- Load API keys safely
- Create an OpenAI client
- Use the modern Responses API
- Understand the response object
- Convert API logic into reusable functions
- Later compare OpenAI, Gemini, Claude and Mistral

In [3]:
#loading api key file using load_dotenv
import os
from dotenv import load_dotenv
from openai import OpenAI
from IPython.display import Markdown, display

load_dotenv()

api_key = os.getenv("OPENAI_API_KEY")

#print(api_key is not None)
client = OpenAI(api_key=api_key)

response = client.responses.create(
model="gpt-4o-mini",
instructions="You are a helpful AI assistant. Answer clearly and briefly.",
input="Explain what an api is in two sentences"
)


#print(response.output_text )
display(Markdown(response.output_text))

An API (Application Programming Interface) is a set of rules and protocols that enables different software applications to communicate with each other. It allows developers to access specific features or data of a service, application, or platform without needing to understand its underlying code.

In [1]:
import sys

print(sys.executable)

c:\Users\sivag\git-projects\week-01-llm-api-foundations\.venv\Scripts\python.exe


## Reusable OpenAI Function

Instead of rewriting the API call every time, create a function that accepts a prompt and returns the generated text.

In [5]:
def ask_openai(prompt: str) -> str:
    response = client.responses.create(
        model="gpt-5-mini",
        instructions="You are a helpful AI assistant. Answer clearly and briefly.",
        input=prompt
    )

    return response.output_text

In [6]:
answer = ask_openai(
    "Explain the difference between AI, Machine Learning and Deep Learning."
)

display(Markdown(answer))

Short answer
- AI (Artificial Intelligence) is the broad field of making machines perform tasks that would require “intelligence” if done by humans.
- Machine Learning (ML) is a subset of AI where systems learn patterns from data to make predictions or decisions instead of being explicitly programmed.
- Deep Learning (DL) is a subset of ML that uses multi-layer neural networks to automatically learn hierarchical features from large amounts of data.

More detail (key distinctions)
- Scope
  - AI: All methods for intelligent behavior — rule-based systems, search, planning, expert systems, ML, etc.
  - ML: Algorithms that improve performance by learning from data (supervised, unsupervised, reinforcement).
  - DL: Neural-network-based ML models with many layers (deep nets).

- How they work
  - AI: Can include hand-coded rules, logic, symbolic reasoning or learned models.
  - ML: Uses explicit algorithms (e.g., linear regression, decision trees, SVMs) that often require manual feature engineering.
  - DL: Learns features automatically end-to-end from raw input using stacked layers of neurons.

- Data & compute needs
  - AI: Varies widely.
  - ML: Often works with moderate-sized structured datasets.
  - DL: Typically needs large datasets and significant compute (GPUs/TPUs) to excel.

- Typical strengths & weaknesses
  - AI: Broadest toolkit; can solve problems without large training sets (e.g., expert systems) but may lack adaptability.
  - ML: Good for many predictive tasks; more interpretable (depending on model) and lighter-weight.
  - DL: State-of-the-art for unstructured data (images, audio, natural language) but less interpretable and resource-intensive.

- Examples
  - AI (non-ML): A rule-based chatbot that follows scripted responses.
  - ML: A random forest classifier predicting loan defaults from tabular features.
  - DL: A convolutional neural network for image recognition or a transformer model for language tasks.

Simple hierarchy: AI ⊇ ML ⊇ DL.

## Claude API

Now we will implement the same basic LLM call using Anthropic's Messages API.

In [18]:
from dotenv import load_dotenv
import os
from anthropic import Anthropic

load_dotenv("../.env", override=True)

claude_api_key = os.getenv("ANTHROPIC_API_KEY")

print("Claude API key loaded:", claude_api_key is not None)



claude_client = Anthropic(
    api_key=claude_api_key
)


Claude API key loaded: True


In [15]:
def ask_claude(prompt: str) -> str:
    response = claude_client.messages.create(
        model="claude-sonnet-4-5",
        max_tokens=300,
        system="You are a helpful AI assistant. Answer clearly and briefly.",
        messages=[
            {
                "role": "user",
                "content": prompt
            }
        ]
    )

    return response.content[0].text

In [19]:
answer = ask_claude(
    "Explain the difference between AI, Machine Learning and Deep Learning."
)

display(Markdown(answer))

# AI, Machine Learning, and Deep Learning

These terms represent increasingly specific subsets of technology:

## **Artificial Intelligence (AI)**
- **Broadest concept**: Any technique that enables computers to mimic human intelligence
- Includes rule-based systems, expert systems, and robotics
- Example: A chess program using predefined rules

## **Machine Learning (ML)**
- **Subset of AI**: Systems that learn from data without explicit programming
- Uses algorithms to find patterns and make predictions
- Example: Email spam filters that improve over time

## **Deep Learning (DL)**
- **Subset of ML**: Uses artificial neural networks with multiple layers
- Inspired by the human brain's structure
- Excels at processing unstructured data (images, speech, text)
- Example: Facial recognition or voice assistants

## **Simple Analogy**
Think of Russian nesting dolls:
- AI is the largest doll (contains everything)
- ML fits inside AI
- DL fits inside ML

Each level is more specialized and powerful for specific tasks, but also requires more data and computational resources.

## OpenAI vs Claude — Same Prompt Comparison

Use the same prompt with both providers and compare:
- API structure
- response style
- clarity
- conciseness

In [20]:
prompt = """
Explain Retrieval-Augmented Generation (RAG) to a junior software engineer.

Cover:
- what problem RAG solves
- the basic architecture
- one real-world use case
"""

In [21]:
openai_answer = ask_openai(prompt)
claude_answer = ask_claude(prompt)

In [22]:
display(Markdown("## OpenAI"))
display(Markdown(openai_answer))

display(Markdown("## Claude"))
display(Markdown(claude_answer))

## OpenAI

Short answer
Retrieval-Augmented Generation (RAG) is a pattern that improves generative LLM answers by first retrieving relevant documents from an external knowledge store and then conditioning the LLM on those documents when generating a response. It helps make outputs more accurate, up-to-date, and grounded in source material.

What problem RAG solves
- LLMs hallucinate facts and have limited context windows — they can invent or forget specifics and they only “know” what was in their training cutoff.
- RAG provides external, up-to-date factual context at inference time so the model can base answers on real documents rather than only its parameters.
- It also scales knowledge (you don’t have to encode a giant knowledge base inside model weights).

Basic architecture and flow (high level)
1. Indexing phase (offline)
   - Split source documents into chunks (e.g., paragraphs or logical blocks).
   - Create dense embeddings for each chunk (embedding model).
   - Store embeddings + metadata in a vector store (FAISS, Milvus, Pinecone, Elastic with vectors, etc.).

2. Query/online phase
   - User issues a query.
   - Embed the query with the same embedding model.
   - Retrieve top-k similar chunks from the vector store (k often 3–10).
   - Optionally rerank retrieved chunks with a cross-encoder for precision.
   - Construct a prompt that includes the retrieved chunks (and instructions) and pass it to the generator LLM (read-and-generate).
   - LLM generates the final answer, ideally citing sources or indicating provenance.

Variants and details
- Retriever type: dense (vector embeddings) vs sparse (BM25) vs hybrid.
- Generator integration: simple concatenation (retrieve-then-read), or Fusion-in-Decoder (FiD) where each doc is separately encoded and fused during decoding for better performance.
- Optional reranker to improve precision.
- You can fine-tune the generator or use few-shot prompting.
- Keep chunk size and number tuned to your model’s context window.

Simple step example
User: “How do I reset my company VPN password?”
1. Query embedded → vector DB returns 5 doc chunks from the IT KB.
2. Prompt assembled: instructions + retrieved chunks.
3. LLM answers: step-by-step reset instructions and cites “IT KB — VPN Reset, doc_id=123”.

Real-world use case (concise)
Enterprise knowledge base / customer support chatbot
- Problem: support agents and customers ask specific, up-to-date product/admin questions (policy, billing, bugs).
- RAG setup:
  - Index product docs, FAQs, internal runbooks, recent tickets.
  - User query retrieves the most relevant runbook/KB paragraphs.
  - LLM generates an answer grounded in those documents and returns citations/links.
- Benefits: more accurate answers, reduced hallucination, ability to surface up-to-date policies without retraining the model, faster agent onboarding, and reduced ticket load.

Practical notes for a junior engineer
- Libraries/services: FAISS, Milvus, Pinecone, Weaviate; OpenAI/other embedding APIs; Hugging Face / Llama2 / GPT for generation.
- Watch for chunking strategy, metadata (source, timestamp), and retrieval evaluation (recall is critical).
- Caveats: RAG reduces but doesn’t eliminate hallucinations; retrieval quality and prompt design still matter; there’s extra latency and infrastructure cost.

That’s the gist: RAG = retrieve relevant documents at query time + generate grounded answers using those documents, giving more accurate and up-to-date outputs than a “closed-book” LLM alone.

## Claude

# Retrieval-Augmented Generation (RAG) Explained

## The Problem RAG Solves

Imagine you're building a chatbot using an LLM like GPT-4. You face two key problems:

1. **Outdated knowledge** - The model was trained months/years ago and doesn't know recent information
2. **No private data** - It doesn't know anything about your company's documents, policies, or internal knowledge

You *could* fine-tune the model, but that's expensive and needs to be redone frequently.

**RAG solves this** by letting the LLM access external information on-demand, without retraining.

## Basic Architecture

RAG works in a simple flow:

```
User Question → Retrieve Relevant Docs → LLM (Question + Docs) → Answer
```

**Step-by-step:**

1. **Index your documents** - Convert documents into embeddings (vector representations) and store them in a vector database
2. **User asks a question** - "What's our return policy?"
3. **Retrieve relevant chunks** - Search the vector DB for the most similar document chunks
4. **Augment the prompt** - Combine the question + retrieved documents into one prompt
5. **Generate answer** - The LLM reads the context and generates an accurate answer

##